# What Earns a Phrase Its Own Token? — Interactive Pretrained Model & Tokenizer Demo
**Author:** Andrew Rober Fouad Samy ([BytesGlue](https://www.bytesglue.com) · [Research Portal](https://andrewrober.github.io) · [GitHub](https://github.com/AndrewRober) · [LinkedIn](https://www.linkedin.com/in/andrew-rober/))

[![Paper PDF](https://img.shields.io/badge/Paper-15--Page%20PDF-b31b1b)](https://andrewrober.github.io/papers/mechanistic-phrase-tokenization.pdf) [![HF Models](https://img.shields.io/badge/%F0%9F%A4%97%20HuggingFace-30M%20%26%20150M%20Weights-ffb000)](https://huggingface.co/AndrewR123/MechanisticPhraseTokenization)

This self-contained notebook runs in **< 45 seconds** on a free Colab CPU/GPU instance:
1. Downloads the official `16,384`-entry **`BT2_AttnDemux_16384`** superword tokenizer & **`B1_ByteBPE_16384`** baseline from HuggingFace Hub.
2. Verifies **exact `0.0` floating-point error** when folding **Compositional-Residual Superwords** ($W_P = \frac{1}{|P|}\sum_{t\in P} W_t + \Delta_P$) into a standard `[V, d]` embedding matrix (`vLLM` / `llama.cpp` ready).
3. Demonstrates **zero-retraining Prefix-Cone Vote Aggregation** curing *Superword Vote-Splitting* while leaving subword models identically unchanged (`0.00%`).
4. Explores **Single-Checkpoint Test-Time Compute Gearshifting** (`Superword High Gear` $\to$ `Subword Low Gear`) on the pretrained **`BT3_FullSynthesis`** Transformer.

In [1]:
!git clone --depth 1 -q https://github.com/AndrewRober/MechanisticPhraseTokenization.git /tmp/mpt
!pip install -q huggingface_hub tokenizers
import sys, torch
sys.path.insert(0, '/tmp/mpt')
from huggingface_hub import snapshot_download
from src.config import TokenizerSpec, ModelSpec
from src.tokenizer_core import PhraseAugmentedTokenizer
from src.breakthroughs import FoldableResidualTransformerLM, PrefixConeDecoder

hf_dir = snapshot_download(
    repo_id='AndrewR123/MechanisticPhraseTokenization',
    allow_patterns=['tokenizers/B1_ByteBPE_16384/*', 'tokenizers/BT2_AttnDemux_16384/*', 'runs/stageC_300M_BT3_FullSynthesis_seed1001/*']
)
bpe_tok = PhraseAugmentedTokenizer.load(f'{hf_dir}/tokenizers/B1_ByteBPE_16384', spec=TokenizerSpec())
bt2_tok = PhraseAugmentedTokenizer.load(f'{hf_dir}/tokenizers/BT2_AttnDemux_16384', spec=TokenizerSpec())
print('Installed MechanisticPhraseTokenization & downloaded tokenizers from AndrewR123/MechanisticPhraseTokenization!')
print(f'B1_ByteBPE vocab: {bpe_tok.vocab_size} | BT2_AttnDemux vocab: {bt2_tok.vocab_size} (14336 shared base + 2048 de-multiplexed superwords)')

Installed MechanisticPhraseTokenization & downloaded tokenizers from AndrewR123/MechanisticPhraseTokenization!
B1_ByteBPE vocab: 16384 | BT2_AttnDemux vocab: 16384 (14336 shared base + 2048 de-multiplexed superwords)


In [2]:
# 1. Compare Byte-BPE vs. Boundary-Guarded Attention-De-Multiplexed Superwords
sample = 'In the midst of rapid climate change, carbon dioxide emissions in New York and North Carolina were analyzed over the course of 2026.'
ids_bpe = bpe_tok.encode(sample, add_eos=False)
ids_bt2 = bt2_tok.encode(sample, add_eos=False)
assert bt2_tok.decode(ids_bt2) == sample
print(f'Sample text ({len(sample.encode())} bytes):')
print(f'  B1_ByteBPE   -> {len(ids_bpe)} tokens | {[bpe_tok.decode([i]) for i in ids_bpe[:10]]}')
print(f'  BT2_AttnDemux-> {len(ids_bt2)} tokens ({(len(ids_bt2)/len(ids_bpe)-1)*100:.1f}% fewer tokens, 100% lossless UTF-8 roundtrip: {bt2_tok.decode(ids_bt2) == sample})')

Sample text (134 bytes):
  B1_ByteBPE   -> 29 tokens | ['In', ' the', ' midst', ' of', ' rapid', ' climate', ' change', ',', ' carbon', ' dioxide', ...]
  BT2_AttnDemux-> 23 tokens (-20.7% fewer tokens, 100% lossless UTF-8 roundtrip: True)


In [3]:
# 2. Verify Exact 0.0 Float Error Folding of Compositional-Residual Superwords
torch.manual_seed(42)
toy_spec = ModelSpec(vocab_size=bt2_tok.vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=512, max_seq_len=64)
foldable = FoldableResidualTransformerLM(toy_spec, bt2_tok.phrase_constituents).eval()
with torch.no_grad():
    foldable.phrase_residual.normal_(0, 0.25)
    x = torch.tensor([ids_bt2[:16]])
    logits_train, _ = foldable(x)
    standard_lm = foldable.fold_to_standard_model().eval()
    logits_deploy, _ = standard_lm(x)
    max_err = (logits_train - logits_deploy).abs().max().item()
print(f'Max absolute logit difference after algebraic folding into standard [V, d] Transformer: {max_err:.6e}')
print('Deployment parameter overhead: 0 extra parameters, 0 extra FLOPs!')

Max absolute logit difference after algebraic folding into standard [V, d] Transformer: 0.000000e+00
Deployment parameter overhead: 0 extra parameters, 0 extra FLOPs!


In [4]:
# 3. Load Pretrained 30.42M BT3_FullSynthesis Checkpoint: Prefix-Cone Decoding & Compute Gearshifting
from src.model import ResearchTransformerLM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lm = ResearchTransformerLM(ModelSpec()).to(device).eval()
ckpt = torch.load(f'{hf_dir}/runs/stageC_300M_BT3_FullSynthesis_seed1001/ckpt_final.pt', map_location=device, weights_only=False)
lm.load_state_dict(ckpt['model'])
cone = PrefixConeDecoder(bt2_tok, device=device)

prompt = 'Over the course of the experiment, researchers in the midst of'
ids_high_gear = [bt2_tok.eos_id] + bt2_tok.encode(prompt, add_eos=False)
ids_low_gear  = [bpe_tok.eos_id] + bpe_tok.encode(prompt, add_eos=False)  # Exact shared 14,336 base subwords!

with torch.no_grad():
    logits_hi, _ = lm(torch.tensor([ids_high_gear], device=device), active_output_vocab=bt2_tok.vocab_size)
    next_cone = cone.argmax_step(logits_hi[:, -1, :]).item()
    logits_lo, _ = lm(torch.tensor([ids_low_gear], device=device), active_output_vocab=14336)
    next_lo = logits_lo[:, -1, :14336].argmax(dim=-1).item()

print('Loaded pretrained 30.42M BT3_FullSynthesis checkpoint (300M unique FineWeb-Edu tokens)!')
print(f'Prompt: {prompt!r}')
print(f'  [High Gear - Superwords] Prompt tokens: {len(ids_high_gear)} | Prefix-Cone Next Token: {bt2_tok.decode([next_cone])!r}')
print(f'  [Low Gear  - Subwords  ] Prompt tokens: {len(ids_low_gear)} | Subword Next Token:     {bt2_tok.decode([next_lo])!r}')

Loaded pretrained 30.42M BT3_FullSynthesis checkpoint (300M unique FineWeb-Edu tokens)!
Prompt: 'Over the course of the experiment, researchers in the midst of'
  [High Gear - Superwords] Prompt tokens: 10 | Prefix-Cone Next Token: ' the'
  [Low Gear  - Subwords  ] Prompt tokens: 13 | Subword Next Token:     ' the'
